# System Prompt Ablation: Conservative vs Base (Reasoning Model)

Comparing ECE, MAE, and STD across domains for conservative and base system prompts using o4-mini (reasoning model).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['OPENESTIMATE_ROOT'] = '/home/ubuntu/alana/openestimate/'
from utils import load_experiment_results

# Load ablation data for all datasets
datasets = ['nhanes', 'pitchbook', 'glassdoor']
dataset_labels = ['NHANES', 'Pitchbook', 'Glassdoor']
results = {d: load_experiment_results(d, 'ablations') for d in datasets}

In [ ]:
# Filter to o4-mini reasoning model, direct protocol, medium temp, base/conservative sysprompts

def filter_sysprompt_comparison(df):
    """Filter to o4-mini reasoning model with direct protocol and medium temp."""
    mask = (
        (df['model'] == 'o4-mini') &
        (df['temperature'] == 'medium') &
        (df['elicitation_protocol'] == 'direct') &
        (df['sysprompt_type'].isin(['base', 'conservative']))
    )
    return df[mask].copy()

filtered = {d: filter_sysprompt_comparison(r) for d, r in results.items()}

# Show counts
for d, df in filtered.items():
    print(f"\n{d.upper()}:")
    print(df.groupby(['sysprompt_type']).size())

In [ ]:
def compute_ece(df):
    """Compute Expected Calibration Error from quartile distribution."""
    q_counts = df['quartile_of_gt'].value_counts(normalize=True) * 100
    ece = sum(abs(q_counts.get(q, 0) - 25) for q in [1, 2, 3, 4]) / 4
    return ece

def compute_raw_metrics(df):
    """Compute raw MAE and STD averages."""
    return {
        'mae': df['abs_error_from_mean'].mean(),
        'std': df['std'].mean()
    }

# Compute metrics for each dataset and sysprompt
metrics_data = []

for d, label in zip(datasets, dataset_labels):
    df = filtered[d]
    for sysprompt in ['base', 'conservative']:
        subset = df[df['sysprompt_type'] == sysprompt]
        if len(subset) == 0:
            continue
        
        ece = compute_ece(subset)
        raw = compute_raw_metrics(subset)
        
        metrics_data.append({
            'dataset': label,
            'sysprompt': sysprompt,
            'ece': ece,
            'mae': raw['mae'],
            'std': raw['std']
        })

metrics_df = pd.DataFrame(metrics_data)
print(metrics_df.to_string(index=False))

In [ ]:
# Create three bar plots: ECE, MAE, STD
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Color palette
colors = {'base': '#1f77b4', 'conservative': '#ff7f0e'}
sysprompts = ['base', 'conservative']
x = np.arange(len(dataset_labels))
width = 0.35

# Plot 1: ECE
ax = axes[0]
for i, sp in enumerate(sysprompts):
    subset = metrics_df[metrics_df['sysprompt'] == sp]
    values = [subset[subset['dataset'] == d]['ece'].values[0] for d in dataset_labels]
    ax.bar(x + (i - 0.5) * width, values, width, label=sp.capitalize(), color=colors[sp])

ax.set_ylabel('ECE (%)')
ax.set_title('Expected Calibration Error (ECE)')
ax.set_xticks(x)
ax.set_xticklabels(dataset_labels)
ax.legend()
ax.set_ylim(0, None)

# Plot 2: MAE
ax = axes[1]
for i, sp in enumerate(sysprompts):
    subset = metrics_df[metrics_df['sysprompt'] == sp]
    values = [subset[subset['dataset'] == d]['mae'].values[0] for d in dataset_labels]
    ax.bar(x + (i - 0.5) * width, values, width, label=sp.capitalize(), color=colors[sp])

ax.set_ylabel('MAE')
ax.set_title('Mean Absolute Error (MAE)')
ax.set_xticks(x)
ax.set_xticklabels(dataset_labels)
ax.legend()
ax.set_ylim(0, None)

# Plot 3: STD
ax = axes[2]
for i, sp in enumerate(sysprompts):
    subset = metrics_df[metrics_df['sysprompt'] == sp]
    values = [subset[subset['dataset'] == d]['std'].values[0] for d in dataset_labels]
    ax.bar(x + (i - 0.5) * width, values, width, label=sp.capitalize(), color=colors[sp])

ax.set_ylabel('STD')
ax.set_title('Standard Deviation (STD)')
ax.set_xticks(x)
ax.set_xticklabels(dataset_labels)
ax.legend()
ax.set_ylim(0, None)

plt.suptitle('System Prompt Comparison: Base vs Conservative (o4-mini)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('analysis_results/sysprompt_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table showing percentage change from base to conservative
print("\nPercentage change (Conservative vs Base):")
print("="*60)

for label in dataset_labels:
    base = metrics_df[(metrics_df['dataset'] == label) & (metrics_df['sysprompt'] == 'base')].iloc[0]
    cons = metrics_df[(metrics_df['dataset'] == label) & (metrics_df['sysprompt'] == 'conservative')].iloc[0]
    
    ece_change = (cons['ece'] - base['ece']) / base['ece'] * 100
    mae_change = (cons['mae'] - base['mae']) / base['mae'] * 100
    std_change = (cons['std'] - base['std']) / base['std'] * 100
    
    print(f"\n{label}:")
    print(f"  ECE: {base['ece']:.2f} -> {cons['ece']:.2f} ({ece_change:+.1f}%)")
    print(f"  MAE: {base['mae']:.4f} -> {cons['mae']:.4f} ({mae_change:+.1f}%)")
    print(f"  STD: {base['std']:.4f} -> {cons['std']:.4f} ({std_change:+.1f}%)")

## Glassdoor: Variables Where Conservative Improved CRPS

Analyzing which variables improved CRPS (Continuous Ranked Probability Score) with the conservative prompt and whether MAE/STD changed. CRPS is a proper scoring rule that captures both accuracy and calibration.

In [ ]:
# Glassdoor: Find variables where CRPS improved with conservative prompt
# and analyze whether MAE/STD changed

from utils import compute_crps_scores

# Compute CRPS for the filtered data
glassdoor_df = filtered['glassdoor'].copy()
glassdoor_df = compute_crps_scores(glassdoor_df)

base = glassdoor_df[glassdoor_df['sysprompt_type'] == 'base']
cons = glassdoor_df[glassdoor_df['sysprompt_type'] == 'conservative']

var_analysis = []
for var in base['variable'].unique():
    b = base[base['variable'] == var]
    c = cons[cons['variable'] == var]
    if len(b) == 0 or len(c) == 0:
        continue
    
    # Metrics
    b_mae = b['abs_error_from_mean'].mean()
    c_mae = c['abs_error_from_mean'].mean()
    b_std = b['std'].mean()
    c_std = c['std'].mean()
    b_crps = b['crps'].mean()
    c_crps = c['crps'].mean()
    
    # CRPS improvement (lower is better)
    crps_improved = c_crps < b_crps
    crps_pct_change = (c_crps - b_crps) / (abs(b_crps) + 1e-10) * 100
    
    mae_pct_change = (c_mae - b_mae) / (b_mae + 1e-10) * 100
    std_pct_change = (c_std - b_std) / (b_std + 1e-10) * 100
    
    var_analysis.append({
        'variable': var,
        'base_crps': b_crps,
        'cons_crps': c_crps,
        'crps_pct_change': crps_pct_change,
        'crps_improved': crps_improved,
        'base_mae': b_mae,
        'cons_mae': c_mae,
        'mae_pct_change': mae_pct_change,
        'base_std': b_std,
        'cons_std': c_std,
        'std_pct_change': std_pct_change
    })

var_df = pd.DataFrame(var_analysis)

# Filter to variables where CRPS improved
improved_df = var_df[var_df['crps_improved']].copy()
improved_df = improved_df.sort_values('crps_pct_change')

print(f"Glassdoor: {len(improved_df)}/{len(var_df)} variables improved CRPS with conservative prompt")
print("="*80)
print("\nVariables with improved CRPS (sorted by CRPS % change):\n")

display_cols = ['variable', 'base_crps', 'cons_crps', 'crps_pct_change', 'mae_pct_change', 'std_pct_change']
print(improved_df[display_cols].to_string(index=False, float_format=lambda x: f'{x:.1f}'))

In [ ]:
# Summary statistics for variables with improved CRPS
print("\nSummary for variables with improved CRPS:")
print("="*60)

# How many had MAE change vs stay similar?
mae_increased = (improved_df['mae_pct_change'] > 15).sum()
mae_decreased = (improved_df['mae_pct_change'] < -15).sum()
mae_similar = len(improved_df) - mae_increased - mae_decreased

std_increased = (improved_df['std_pct_change'] > 15).sum()
std_decreased = (improved_df['std_pct_change'] < -15).sum()
std_similar = len(improved_df) - std_increased - std_decreased

print(f"\nMAE changes (threshold: 15%):")
print(f"  Increased: {mae_increased} ({mae_increased/len(improved_df)*100:.0f}%)")
print(f"  Decreased: {mae_decreased} ({mae_decreased/len(improved_df)*100:.0f}%)")
print(f"  Similar:   {mae_similar} ({mae_similar/len(improved_df)*100:.0f}%)")

print(f"\nSTD changes (threshold: 15%):")
print(f"  Increased: {std_increased} ({std_increased/len(improved_df)*100:.0f}%)")
print(f"  Decreased: {std_decreased} ({std_decreased/len(improved_df)*100:.0f}%)")
print(f"  Similar:   {std_similar} ({std_similar/len(improved_df)*100:.0f}%)")

print(f"\nMean changes across variables with improved CRPS:")
print(f"  MAE change: {improved_df['mae_pct_change'].mean():+.1f}%")
print(f"  STD change: {improved_df['std_pct_change'].mean():+.1f}%")
print(f"  CRPS change: {improved_df['crps_pct_change'].mean():+.1f}%")

In [ ]:
# Scatter plot: MAE change vs STD change for variables with improved CRPS
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(improved_df['mae_pct_change'], improved_df['std_pct_change'], 
           s=80, alpha=0.7, c='steelblue', edgecolors='black')

# Add reference lines
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.axhline(15, color='red', linestyle=':', alpha=0.5)
ax.axhline(-15, color='red', linestyle=':', alpha=0.5)
ax.axvline(15, color='red', linestyle=':', alpha=0.5)
ax.axvline(-15, color='red', linestyle=':', alpha=0.5)

ax.set_xlabel('MAE % Change (Conservative vs Base)')
ax.set_ylabel('STD % Change (Conservative vs Base)')
ax.set_title('Glassdoor: Variables with Improved CRPS\n(Red lines = 15% threshold)')

# Add quadrant labels
ax.text(0.95, 0.95, 'MAE↑ STD↑', transform=ax.transAxes, ha='right', va='top', fontsize=9, color='gray')
ax.text(0.05, 0.95, 'MAE↓ STD↑', transform=ax.transAxes, ha='left', va='top', fontsize=9, color='gray')
ax.text(0.05, 0.05, 'MAE↓ STD↓', transform=ax.transAxes, ha='left', va='bottom', fontsize=9, color='gray')
ax.text(0.95, 0.05, 'MAE↑ STD↓', transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('analysis_results/glassdoor_crps_improvement_scatter.png', dpi=150, bbox_inches='tight')
plt.show()